# ARIMA Modeling for NZD/USD Exchange Rates

This notebook builds initial ARIMA-style forecasting models for the NZD/USD exchange rate.

The previous notebook showed that the raw exchange-rate level and log exchange-rate level are likely nonstationary, while the daily log return series appears stationary. Based on those findings, this notebook starts with simple ARIMA-style models and compares their performance against the naïve and drift baselines.

## 1. Modeling objective

The goal of this notebook is to test whether ARIMA-style models can improve on the simple baseline forecasts built earlier.

The main modeling options are:

1. Model the log exchange-rate level with differencing.
2. Model the daily log return series directly.

Because the naïve baseline is difficult to beat in exchange-rate forecasting, any ARIMA model should be evaluated against the baseline MAE and RMSE values from the previous notebook.

## 2. Load processed dataset

This section loads the cleaned and transformed NZD/USD dataset created earlier in the project.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

cleaned_data = pd.read_csv('/Users/masonwang/Documents/Projects/Time Series Analysis/NZD-USD-Exchange-Rate-Forecaster/data/processed/cleaned_nzd_usd.csv')
cleaned_data['observation_date'] = pd.to_datetime(cleaned_data['observation_date'])
cleaned_data

,observation_date,DEXUSNZ,log_nzd_usd,nzd_usd_return
0,2016-06-06,0.6931,-0.366581,NaN
1,2016-06-07,0.6981,-0.359393,0.007188
2,2016-06-08,0.7024,-0.353252,0.006141
3,2016-06-09,0.7133,-0.337853,0.015399
4,2016-06-10,0.7069,-0.346866,-0.009013
...,...,...,...,...
2493,2026-06-01,0.5931,-0.522392,-0.010232
2494,2026-06-02,0.5933,-0.522055,0.000337
2495,2026-06-03,0.5871,-0.532560,-0.010505
2496,2026-06-04,0.5879,-0.531198,0.001362


The processed NZD/USD dataset was loaded and sorted chronologically. This dataset contains the exchange-rate level, log exchange rate, and daily log return series needed for ARIMA modeling.

## 3. Train-test split

This section creates a time-based train-test split. For time-series forecasting, the data should not be randomly split because that would allow future information to leak into the training process.

The training set contains earlier observations, while the test set contains later observations. This better reflects a real forecasting workflow, where models are trained on the past and evaluated on future data.

This notebook starts with `log_nzd_usd` because ARIMA can difference the log exchange-rate level internally through the `d` parameter.

In [28]:
split_index = int(len(cleaned_data) * 0.8)

train_data = cleaned_data.iloc[:split_index].copy()
test_data = cleaned_data.iloc[split_index:].copy()

log_train = train_data[['observation_date', 'log_nzd_usd']].copy()
log_test = test_data[['observation_date', 'log_nzd_usd']].copy()

print(f'Training observations: {len(log_train)}')
print(f'Testing observations: {len(log_test)}')
print(f'Training start date: {log_train['observation_date'].min()}')
print(f'Training end date: {log_train['observation_date'].max()}')
print(f'Testing start date: {log_test['observation_date'].min()}')
print(f'Testing end date: {log_test['observation_date'].max()}')

Training observations: 1998
Testing observations: 500
Training start date: 2016-06-06 00:00:00
Training end date: 2024-06-05 00:00:00
Testing start date: 2024-06-06 00:00:00
Testing end date: 2026-06-05 00:00:00


In [29]:
y_train = train_data['log_nzd_usd']
y_test = test_data['log_nzd_usd']

The dataset was split chronologically, with the first 80% of observations used for training and the final 20% reserved for testing. This avoids look-ahead bias and allows the model to be evaluated on future observations it did not see during training.

The `y_train` and `y_test` objects contain the one-dimensional log exchange-rate series that will be used for ARIMA model fitting and evaluation.

## 4. Fit initial ARIMA model

This section fits an initial ARIMA model to the training data. Based on the previous stationarity analysis, the log exchange-rate level requires differencing, so the first model uses an ARIMA specification with `d = 1`.

In [30]:
from statsmodels.tsa.arima.model import ARIMA
arima_010_model = ARIMA(y_train, order=(0, 1, 0))
arima_010_results = arima_010_model.fit()

arima_010_results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                               SARIMAX Results                                
==============================================================================
Dep. Variable:            log_nzd_usd   No. Observations:                 1998
Model:                 ARIMA(0, 1, 0)   Log Likelihood                7227.541
Date:                Wed, 17 Jun 2026   AIC                         -14453.082
Time:                        19:51:44   BIC                         -14447.482
Sample:                             0   HQIC                        -14451.025
                               - 1998                                         
Covariance Type:                  opg                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
sigma2      4.204e-05      1e-06     41.882      0.000    4.01e-05     4.4e-05
===================================================================================
Ljung-Box (L1) (Q):                   0.42   Jarque-Bera (JB):               212.87
Prob(Q):                              0.52   Prob(JB):                         0.00
Heteroskedasticity (H):               1.50   Skew:                            -0.28
Prob(H) (two-sided):                  0.00   Kurtosis:                         4.50
===================================================================================

Warnings:
[1] Covariance matrix calculated using the outer product of gradients (complex-step).
"""

The initial `ARIMA(0, 1, 0)` model was fitted to the log NZD/USD exchange-rate series. Because this model has no autoregressive or moving-average terms, it acts as a random-walk model on the log exchange rate.

The model summary shows that the only estimated parameter is `sigma2`, which represents the variance of the model errors. The Ljung-Box test does not provide strong evidence of residual autocorrelation at lag 1, suggesting that the random-walk specification captures much of the short-term dependence in the differenced log series.

However, the Jarque-Bera and heteroskedasticity tests suggest that the residuals are not normally distributed and have changing variance over time. This is common in financial exchange-rate data, where volatility tends to cluster.

The AIC and BIC values are not meaningful on their own, but they will be useful for comparing this model against more complex ARIMA specifications fitted on the same training data.

## 5. Generate test-set forecasts

This section generates forecasts from the initial `ARIMA(0, 1, 0)` model over the test period. The forecasts are first produced on the log exchange-rate scale, then converted back to the original NZD/USD exchange-rate scale so they can be compared with the baseline models.

In [31]:
forecast_steps = len(y_test)

arima_010_log_forecast = arima_010_results.forecast(steps=forecast_steps)

arima_010_forecast_table = pd.DataFrame({
    'observation_date': test_data['observation_date'].values,
    'actual_log_nzd_usd': y_test.values,
    'forecast_log_nzd_usd': arima_010_log_forecast.values
})

arima_010_forecast_table.head()

,observation_date,actual_log_nzd_usd,forecast_log_nzd_usd
0,2024-06-06,-0.478520,-0.480296
1,2024-06-07,-0.492167,-0.480296
2,2024-06-10,-0.490043,-0.480296
3,2024-06-11,-0.489064,-0.480296
4,2024-06-12,-0.474976,-0.480296


The ARIMA forecasts were generated over the test period. Because the model was fitted to the log exchange-rate series, the forecasts were first produced on the log scale and then exponentiated back to the original NZD/USD exchange-rate scale.

This conversion is necessary because the baseline models were evaluated using the original exchange-rate scale. Evaluating all models on the same scale allows for a fair comparison of MAE and RMSE.

In [32]:
arima_010_forecast_table['actual_nzd_usd'] = np.exp(arima_010_forecast_table['actual_log_nzd_usd'])
arima_010_forecast_table['forecast_nzd_usd'] = np.exp(arima_010_forecast_table['forecast_log_nzd_usd'])

arima_010_forecast_table.head()

,observation_date,actual_log_nzd_usd,forecast_log_nzd_usd,actual_nzd_usd,forecast_nzd_usd
0,2024-06-06,-0.478520,-0.480296,0.6197,0.6186
1,2024-06-07,-0.492167,-0.480296,0.6113,0.6186
2,2024-06-10,-0.490043,-0.480296,0.6126,0.6186
3,2024-06-11,-0.489064,-0.480296,0.6132,0.6186
4,2024-06-12,-0.474976,-0.480296,0.6219,0.6186


The `ARIMA(0, 1, 0)` model produces a flat forecast over the test period. This is expected because the model is equivalent to a random walk without drift. The model uses the final observed training value as the forecast for future periods.

The forecasts were converted from the log scale back to the original NZD/USD exchange-rate scale using exponentiation.

## 6. Evaluate ARIMA performance

This section evaluates the initial ARIMA model using MAE and RMSE on the original NZD/USD exchange-rate scale. Evaluating on the original scale allows the ARIMA model to be compared with the baseline models.

In [ ]:
arima_010_forecast_table['forecast_error'] = (arima_010_forecast_table['actual_nzd_usd'] - arima_010_forecast_table['forecast_nzd_usd'])

arima_010_mae = arima_010_forecast_table['forecast_error'].abs().mean()
arima_010_rmse = np.sqrt((arima_010_forecast_table['forecast_error'] ** 2).mean())

print(f'ARIMA(0, 1, 0) MAE: {arima_010_mae:.6f}')
print(f'ARIMA(0, 1, 0) RMSE: {arima_010_rmse:.6f}')

ARIMA(0, 1, 0) MAE: 0.029734
ARIMA(0, 1, 0) RMSE: 0.033495


In [34]:
arima_results_table = pd.DataFrame({
    'Model': ['ARIMA(0, 1, 0)'],
    'MAE': [arima_010_mae],
    'RMSE': [arima_010_rmse]
})

arima_results_table

,Model,MAE,RMSE
0,"ARIMA(0, 1, 0)",0.029734,0.033495


The initial ARIMA model was evaluated using MAE and RMSE on the original NZD/USD exchange-rate scale. These metrics measure the average size of the forecast errors, with RMSE penalizing larger errors more heavily than MAE.

Because this forecast is generated across the full test period from a single fitted model, it should be compared against baselines computed over the same test window.

## 7. Compare against test-period baselines

This section compares the initial ARIMA model against baseline forecasts over the same test period.

The earlier baseline notebook evaluated one-step-ahead forecasts across the full dataset. For a fair comparison here, the baseline forecasts are recalculated using only information available at the end of the training period.

In [35]:
last_train_value = train_data['DEXUSNZ'].iloc[-1]
average_train_change = train_data['DEXUSNZ'].diff().mean()

baseline_test_table = pd.DataFrame({
    'observation_date': test_data['observation_date'].values,
    'actual_nzd_usd': test_data['DEXUSNZ'].values
})

baseline_test_table['forecast_horizon'] = np.arange(1, len(baseline_test_table) + 1)

baseline_test_table['naive_forecast'] = last_train_value
baseline_test_table['drift_forecast'] = (
    last_train_value 
    + baseline_test_table['forecast_horizon'] * average_train_change
)

baseline_test_table.head()

,observation_date,actual_nzd_usd,forecast_horizon,naive_forecast,drift_forecast
0,2024-06-06,0.6197,1,0.6186,0.618563
1,2024-06-07,0.6113,2,0.6186,0.618525
2,2024-06-10,0.6126,3,0.6186,0.618488
3,2024-06-11,0.6132,4,0.6186,0.618451
4,2024-06-12,0.6219,5,0.6186,0.618413


In [36]:
baseline_test_table['naive_error'] = (
    baseline_test_table['actual_nzd_usd'] 
    - baseline_test_table['naive_forecast']
)

baseline_test_table['drift_error'] = (
    baseline_test_table['actual_nzd_usd'] 
    - baseline_test_table['drift_forecast']
)

test_naive_mae = baseline_test_table['naive_error'].abs().mean()
test_naive_rmse = np.sqrt((baseline_test_table['naive_error'] ** 2).mean())

test_drift_mae = baseline_test_table['drift_error'].abs().mean()
test_drift_rmse = np.sqrt((baseline_test_table['drift_error'] ** 2).mean())

print(f'Test-period naive MAE: {test_naive_mae:.6f}')
print(f'Test-period naive RMSE: {test_naive_rmse:.6f}')
print(f'Test-period drift MAE: {test_drift_mae:.6f}')
print(f'Test-period drift RMSE: {test_drift_rmse:.6f}')

Test-period naive MAE: 0.029734
Test-period naive RMSE: 0.033495
Test-period drift MAE: 0.020696
Test-period drift RMSE: 0.025189


In [37]:
model_comparison_table = pd.DataFrame({
    'Model': [
        'Naive baseline',
        'Drift baseline',
        'ARIMA(0, 1, 0)'
    ],
    'MAE': [
        test_naive_mae,
        test_drift_mae,
        arima_010_mae
    ],
    'RMSE': [
        test_naive_rmse,
        test_drift_rmse,
        arima_010_rmse
    ]
})

model_comparison_table

,Model,MAE,RMSE
0,Naive baseline,0.029734,0.033495
1,Drift baseline,0.020696,0.025189
2,"ARIMA(0, 1, 0)",0.029734,0.033495


The ARIMA model is compared against naïve and drift baselines over the same test period. This ensures that all models are evaluated on the same observations and with the same information cutoff at the end of the training period.

## 8. Test additional ARIMA specifications

This section fits several additional ARIMA models to test whether adding autoregressive or moving-average terms improves forecast performance.

The initial `ARIMA(0, 1, 0)` model is equivalent to a random walk. Additional models such as `ARIMA(1, 1, 0)`, `ARIMA(0, 1, 1)`, and `ARIMA(1, 1, 1)` allow the model to capture short-term autocorrelation patterns in the differenced log exchange-rate series.

In [38]:
candidate_orders = [
    (0, 1, 0),
    (1, 1, 0),
    (0, 1, 1),
    (1, 1, 1),
    (2, 1, 0),
    (0, 1, 2)
]

arima_candidate_results = []

for order in candidate_orders:
    model = ARIMA(y_train, order=order)
    results = model.fit()

    log_forecast = results.forecast(steps=len(y_test))
    forecast_nzd_usd = np.exp(log_forecast)

    actual_nzd_usd = test_data['DEXUSNZ'].values
    forecast_error = actual_nzd_usd - forecast_nzd_usd.values

    mae = np.abs(forecast_error).mean()
    rmse = np.sqrt((forecast_error ** 2).mean())

    arima_candidate_results.append({
        'Model': f'ARIMA{order}',
        'AIC': results.aic,
        'BIC': results.bic,
        'MAE': mae,
        'RMSE': rmse
    })

arima_candidate_table = pd.DataFrame(arima_candidate_results)

arima_candidate_table.sort_values('RMSE')

,Model,AIC,BIC,MAE,RMSE
3,"ARIMA(1, 1, 1)",-14447.781381,-14430.983177,0.029702,0.033464
5,"ARIMA(0, 1, 2)",-14451.045968,-14434.247764,0.029706,0.033468
4,"ARIMA(2, 1, 0)",-14451.193514,-14434.395310,0.029711,0.033474
2,"ARIMA(0, 1, 1)",-14451.520498,-14440.321695,0.029719,0.033480
1,"ARIMA(1, 1, 0)",-14451.497803,-14440.299000,0.029720,0.033481
0,"ARIMA(0, 1, 0)",-14453.081544,-14447.482143,0.029734,0.033495
